# Image Data Augmentation Research Lab — Google Colab

This notebook is the reproducible Google Colab companion to the GitHub project.

**Reference:** Shorten & Khoshgoftaar (2019), *A survey on Image Data Augmentation for Deep Learning*, Journal of Big Data, 6, 60.

The notebook demonstrates image-space augmentation, stochastic policies, sensitivity analysis and controlled variant generation. It does not claim to reproduce the survey's GAN, adversarial-training or meta-learning experiments.

In [ ]:
!pip -q install Pillow numpy pandas matplotlib

In [ ]:
from google.colab import files
from PIL import Image, ImageEnhance, ImageFilter
import io, numpy as np, pandas as pd, matplotlib.pyplot as plt

uploaded = files.upload()
name = next(iter(uploaded))
original = Image.open(io.BytesIO(uploaded[name])).convert('RGB')
print('Image:', name, '| Size:', original.size)
display(original)

## 1. Geometric augmentation

In [ ]:
rotation = original.rotate(25, expand=True, fillcolor='white')
flip = original.transpose(Image.Transpose.FLIP_LEFT_RIGHT)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(original); ax[0].set_title('Original')
ax[1].imshow(flip); ax[1].set_title('Horizontal flip')
ax[2].imshow(rotation); ax[2].set_title('Rotation')
for a in ax: a.axis('off')
plt.show()

## 2. Colour-space augmentation

In [ ]:
brightness = ImageEnhance.Brightness(original).enhance(1.3)
contrast = ImageEnhance.Contrast(original).enhance(1.3)
saturation = ImageEnhance.Color(original).enhance(0.7)

fig, ax = plt.subplots(1, 4, figsize=(18, 5))
for a, im, title in zip(ax, [original, brightness, contrast, saturation], ['Original','Brightness','Contrast','Saturation']):
    a.imshow(im); a.set_title(title); a.axis('off')
plt.show()

## 3. Random erasing

In [ ]:
def random_erasing(img, area_percent=10, seed=42):
    rng = np.random.default_rng(seed)
    arr = np.array(img).copy()
    h, w, _ = arr.shape
    target = max(1, int(h*w*area_percent/100))
    ratio = float(rng.uniform(0.5, 2.0))
    eh = max(1, min(h, int(np.sqrt(target*ratio))))
    ew = max(1, min(w, int(np.sqrt(target/ratio))))
    y = int(rng.integers(0, max(1, h-eh+1)))
    x = int(rng.integers(0, max(1, w-ew+1)))
    arr[y:y+eh, x:x+ew] = 0
    return Image.fromarray(arr)

erased = random_erasing(original, 12, 2026)
display(erased)

## 4. Pixel-level diagnostics

In [ ]:
def mse(a,b):
    x=np.asarray(a.resize(b.size),dtype=np.float32); y=np.asarray(b,dtype=np.float32)
    return float(np.mean((x-y)**2))

def mae(a,b):
    x=np.asarray(a.resize(b.size),dtype=np.float32); y=np.asarray(b,dtype=np.float32)
    return float(np.mean(np.abs(x-y)))

def psnr(a,b):
    value=mse(a,b)
    return float('inf') if value==0 else float(10*np.log10(255**2/value))

results = pd.DataFrame([{
    'augmentation':'random erasing',
    'MSE':mse(original,erased),
    'MAE':mae(original,erased),
    'PSNR_dB':psnr(original,erased)
}])
results

## Research note

Pixel-level difference is a diagnostic of image change, not evidence of improved model generalisation. A proper performance experiment should use a fixed architecture, controlled train/validation/test split, repeated seeds and task-specific metrics.